In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [12]:
%%writefile vector.cu
#include<stdio.h>
#include<cuda_runtime.h>
#define N 5

__global__ void add(int *A, int *B, int *C) {
    int i = threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }

}
int main(){

    int A[N]={1,2,3,4,5};
    int B[N]={6,7,8,9,10};
    int C[N];
    int *d_A,*d_B,*d_C;
    cudaMalloc((void**)&d_A,N*sizeof(int));
    cudaMalloc((void**)&d_B,N*sizeof(int));
    cudaMalloc((void**)&d_C,N*sizeof(int));
    cudaMemcpy(d_A,A,N*sizeof(int),cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,N*sizeof(int),cudaMemcpyHostToDevice);
    add<<<1,N>>>(d_A,d_B,d_C);
    cudaDeviceSynchronize();
    cudaMemcpy(C,d_C,N*sizeof(int),cudaMemcpyDeviceToHost);
    for(int i=0;i<N;i++){
        printf("%d ",C[i]);
    }
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    return 0;  
    
    

    
    
    
}



Overwriting vector.cu


In [13]:
!nvcc vector.cu -o vector

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [14]:
!./vector

7 9 11 13 15 

In [16]:
%%writefile reduction.cu
#include <stdio.h>
#include <cuda_runtime.h>
#define N 8
// CUDA kernel for parallel reduction
__global__ void reduce(int *A) {
    int i = threadIdx.x;
    // Add pairs of elements
    for (int s = 1; s < N; s *= 2) {
        if (i % (2 * s) == 0 && i + s < N) {
            A[i] += A[i + s];
        }
        __syncthreads(); // Synchronize threads
    }
}
int main() {
    int A[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    int *d_A;
    // Allocate GPU memory
    cudaMalloc((void**)&d_A, N * sizeof(int));
    // Copy array to GPU
    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    // Launch 1 block with 8 threads
    reduce<<<1, N>>>(d_A);
    cudaDeviceSynchronize();
    // Copy final sum back to CPU
    cudaMemcpy(A, d_A, N * sizeof(int), cudaMemcpyDeviceToHost);
    printf("Sum = %d\n", A[0]);
    // Free GPU memory
    cudaFree(d_A);
    return 0;
}

Overwriting reduction.cu


In [17]:
!nvcc reduction.cu -o reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [18]:
!./reduction

Sum = 36


In [23]:
%%writefile inclusive.cu
#include <stdio.h>
#include <cuda_runtime.h>
#define N 5
__global__ void prefixSum(int *A, int *B) {
    int i = threadIdx.x;
    int sum = 0;
    for (int j = 0; j <= i; j++) {
        sum += A[j];
    }
    B[i] = sum;
}
int main() {
    int A[N] = {1, 2, 3, 4, 5};
    int B[N];
    int *d_A, *d_B;
    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    prefixSum<<<1, N>>>(d_A, d_B);
    cudaDeviceSynchronize();
    cudaMemcpy(B, d_B, N * sizeof(int), cudaMemcpyDeviceToHost);
    for (int i = 0; i < N; i++)
        printf("%d ", B[i]);
    printf("\n");
    cudaFree(d_A);
    cudaFree(d_B);
    return 0;
}

Overwriting inclusive.cu


In [24]:
!nvcc inclusive.cu -o inclusive

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!./inclusive

1 3 6 10 15 


In [14]:
%%writefile list.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 4

__global__ void listRanking(int *next, int *rank) {
    int i = threadIdx.x;
    int current = 0;
    int count = 0;

    while (current != -1 && current != i) {
        current = next[current];
        count++;
    }

    rank[i] = count;
}

int main() {
    int next[N] = {1, 2, 3, -1};
    int rank[N];

    int *d_next, *d_rank;

    cudaMalloc((void**)&d_next, N * sizeof(int));
    cudaMalloc((void**)&d_rank, N * sizeof(int));

    cudaMemcpy(d_next, next, N * sizeof(int),
               cudaMemcpyHostToDevice);

    listRanking<<<1, N>>>(d_next, d_rank);

    cudaDeviceSynchronize();

    cudaMemcpy(rank, d_rank, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("Node ranks: ");
    for (int i = 0; i < N; i++)
        printf("%d ", rank[i]);

    cudaFree(d_next);
    cudaFree(d_rank);

    return 0;
}

Overwriting list.cu


In [15]:
!nvcc list.cu -o list

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [16]:
!./list

Node ranks: 0 1 2 3 

In [18]:
%%writefile sparse.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 9

__global__ void add(int *A, int *B, int *C) {
    int i = threadIdx.x;

    if (i < N)
        C[i] = A[i] + B[i];
}

int main() {
    int A[N] = {1, 0, 0, 0, 5, 0, 0, 0, 9};
    int B[N] = {0, 2, 0, 0, 0, 3, 0, 0, 4};
    int C[N];

    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, N * sizeof(int));
    cudaMalloc((void**)&d_B, N * sizeof(int));
    cudaMalloc((void**)&d_C, N * sizeof(int));

    cudaMemcpy(d_A, A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N * sizeof(int), cudaMemcpyHostToDevice);

    add<<<1, N>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, N * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Result matrix:\n");
    for (int i = 0; i < N; i++) {
        printf("%d ", C[i]);
        if ((i + 1) % 3 == 0)
            printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting sparse.cu


In [19]:
!nvcc sparse.cu -o sparse

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!./sparse

Result matrix:
1 2 0 
0 5 3 
0 0 13 


In [23]:
%%writefile sort.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 8

__global__ void sort(int *A, int phase) {
    int i = 2 * threadIdx.x + (phase % 2);

    if (i + 1 < N) {
        if (A[i] > A[i + 1]) {
            int temp = A[i];
            A[i] = A[i + 1];
            A[i + 1] = temp;
        }
    }
}

int main() {
    int A[N] = {8, 3, 7, 4, 2, 6, 1, 5};
    int *d_A;

    cudaMalloc((void**)&d_A, N * sizeof(int));

    cudaMemcpy(d_A, A, N * sizeof(int),
               cudaMemcpyHostToDevice);

    for (int phase = 0; phase < N; phase++) {
        sort<<<1, N / 2>>>(d_A, phase);
        cudaDeviceSynchronize();
    }

    cudaMemcpy(A, d_A, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("Sorted array: ");
    for (int i = 0; i < N; i++)
        printf("%d ", A[i]);

    cudaFree(d_A);

    return 0;
}

Overwriting sort.cu


In [24]:
!nvcc sort.cu -o sort

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [25]:
!./sort

Sorted array: 1 2 3 4 5 6 7 8 